# UCL Match & Champion Predictor (2026-27)

Predicts **the next Champions League matches** (home win / draw / away win) and **the champion of Europe** using:

| Input | Source |
|---|---|
| Domestic league results (11 major leagues + AUT/DNK/NOR/SUI), incl. shots on target | football-data.co.uk |
| UCL results 2017-18 → 2026-27 (incl. the current season's fixtures) | fixturedownload.com |
| Club ratings | **Our own Elo** from every domestic + UCL match, **plus ClubElo** ratings scraped from clubelo.com (the ClubElo API is down) |

**Pipeline**
1. Download data, normalise team names.
2. Walk through every match chronologically, recording **pre-match** features (no leakage): Elo, domestic form (last 10), season PPG, shot-on-target share, league strength, UCL experience in the previous 3 seasons.
3. Train several models (Elo-only logistic, full logistic, gradient boosting; UCL-only vs UCL+domestic training). Pick by validation log-loss on 2023-24, report honest test scores on 2024-25 → 2026-27.
4. Fit a Poisson goals model (needed for scorelines, goal difference and two-leg aggregates).
5. Predict the next matchday, then **Monte Carlo the rest of the season** (league phase → play-offs → R16 → QF → SF → final) to get each club's title probability.

▶ **Runtime → Run all.** CPU runtime is fine (no GPU needed); total runtime is a few minutes.

## 0. (Optional) mount Google Drive
Drive keeps the downloaded data, the ClubElo cache and the outputs between Colab sessions. If you decline or mounting isn't possible, everything goes on the Colab disk instead.

In [ ]:
import os
IN_COLAB = os.path.isdir('/content')
USE_DRIVE = True
BASE_DIR = '/content' if IN_COLAB else '.'
if IN_COLAB and USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        BASE_DIR = '/content/drive/MyDrive/ucl_predictor'
    except Exception as e:
        print(f'Drive not mounted ({type(e).__name__}: {e}) - using the Colab disk instead')
os.makedirs(BASE_DIR, exist_ok=True)
print('data / cache / outputs live under', BASE_DIR)

In [ ]:
import os, io, json, warnings
from collections import defaultdict, deque
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, requests
import matplotlib.pyplot as plt
from scipy.stats import poisson
from sklearn.linear_model import LogisticRegression, PoissonRegressor
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import log_loss, accuracy_score

try:
    from IPython.display import display
except ImportError:
    display = print
warnings.filterwarnings('ignore')
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)

IN_COLAB = os.path.isdir('/content')
BASE_DIR = globals().get('BASE_DIR', '/content' if IN_COLAB else '.')
DATA_DIR = os.path.join(BASE_DIR, 'data')
OUT_DIR  = os.path.join(BASE_DIR, 'outputs')
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

CURRENT_SEASON = 2026                       # 2026-27 season
UCL_YEARS  = list(range(2017, CURRENT_SEASON + 1))
FD_SEASONS = [f'{y%100:02d}{(y+1)%100:02d}' for y in range(2016, CURRENT_SEASON + 1)]
MAIN_LEAGUES  = ['E0', 'SP1', 'D1', 'I1', 'F1', 'N1', 'P1', 'B1', 'T1', 'G1', 'SC0']
EXTRA_LEAGUES = ['AUT', 'DNK', 'NOR', 'SWZ']
BURN_IN = pd.Timestamp('2017-07-01')       # first season used only to warm up Elo
N_SIMS = 20000
SEED = 42

## 1. Download data

In [ ]:
def fetch(url, fname):
    path = os.path.join(DATA_DIR, fname)
    is_current = FD_SEASONS[-1] in fname or fname == f'ucl_{CURRENT_SEASON}.json' or fname.startswith('NEW_')
    if is_current or not (os.path.exists(path) and os.path.getsize(path) > 200):
        try:
            r = requests.get(url, timeout=60, headers={'User-Agent': 'ucl-predictor/1.0'})
            r.raise_for_status()
            with open(path, 'wb') as fh:
                fh.write(r.content)
        except Exception as e:
            return f'skip {fname}: {e}'
    return None

jobs  = [(f'https://fixturedownload.com/feed/json/champions-league-{y}', f'ucl_{y}.json') for y in UCL_YEARS]
jobs += [(f'https://www.football-data.co.uk/mmz4281/{s}/{lg}.csv', f'{lg}_{s}.csv')
         for lg in MAIN_LEAGUES for s in FD_SEASONS]
jobs += [(f'https://www.football-data.co.uk/new/{lg}.csv', f'NEW_{lg}.csv') for lg in EXTRA_LEAGUES]

with ThreadPoolExecutor(8) as ex:
    msgs = [m for m in ex.map(lambda j: fetch(*j), jobs) if m]
print(f'{len(jobs) - len(msgs)}/{len(jobs)} files ready'); print(*msgs, sep='\n')

## 2. Load & normalise
football-data.co.uk names are canonical; UCL names (which vary across seasons) are mapped onto them.

In [ ]:
UCL_ALIASES = {
    'AEK Athens': 'AEK', 'Athletic Club': 'Ath Bilbao',
    'Atleti': 'Ath Madrid', 'Atlético': 'Ath Madrid', 'Atlético Madrid': 'Ath Madrid', 'Atlético de Madrid': 'Ath Madrid',
    'B. Dortmund': 'Dortmund', 'Borussia Dortmund': 'Dortmund',
    'Bayern': 'Bayern Munich', 'Bayern München': 'Bayern Munich',
    'Bodø/Glimt': 'Bodo/Glimt', 'Braga': 'Sp Braga', 'Copenhagen': 'FC Copenhagen', 'FC Porto': 'Porto',
    'Fenerbahçe': 'Fenerbahce', 'Frankfurt': 'Ein Frankfurt', 'Internazionale': 'Inter',
    'Istanbul Basaksehir': 'Buyuksehyr', 'LOSC': 'Lille', 'Leipzig': 'RB Leipzig',
    'Man Utd': 'Man United', 'Man. United': 'Man United', 'Manchester United': 'Man United',
    'Man. City': 'Man City', 'Manchester City': 'Man City', 'Mönchengladbach': "M'gladbach",
    'Olympiacos': 'Olympiakos', 'PSV': 'PSV Eindhoven', 'Paris': 'Paris SG', 'Paris Saint-Germain': 'Paris SG',
    'Real Betis': 'Betis', 'Real Sociedad': 'Sociedad', 'Schalke': 'Schalke 04', 'Sporting CP': 'Sp Lisbon',
    'Tottenham Hotspur': 'Tottenham', 'Union SG': 'St. Gilloise',
    # clubs without domestic data: just merge spelling variants
    'CSKA Moskva': 'CSKA Moscow', 'Crvena zvezda': 'Crvena Zvezda', 'GNK Dinamo': 'Dinamo Zagreb',
    'Shakhtar Donetsk': 'Shakhtar', 'Malmö': 'Malmo', 'Ferencváros': 'Ferencvaros',
}
canon = lambda n: UCL_ALIASES.get(n.strip(), n.strip())

def load_ucl():
    rows = []
    for y in UCL_YEARS:
        try:
            data = json.load(open(os.path.join(DATA_DIR, f'ucl_{y}.json'), encoding='utf-8'))
        except Exception:
            print('no UCL data for', y); continue
        league_rounds = 8 if y >= 2024 else 6        # league phase (2024+) / group stage
        final_round   = 17 if y >= 2024 else 13
        for m in data:
            date = pd.Timestamp(m['DateUtc'].replace('Z', '')).normalize()
            rnd = int(m['RoundNumber'])
            neutral = rnd == final_round or (y == 2019 and date >= pd.Timestamp('2020-08-10'))  # Lisbon bubble
            rows.append(dict(date=date, comp='UCL', season_id=f'UCL-{y}', round=rnd,
                             home=canon(m['HomeTeam']), away=canon(m['AwayTeam']),
                             hg=m['HomeTeamScore'], ag=m['AwayTeamScore'], hst=np.nan, ast=np.nan,
                             neutral=bool(neutral), is_ko=rnd > league_rounds, is_final=rnd == final_round,
                             winner=canon(m['Winner']) if m.get('Winner') else None,
                             played=m['HomeTeamScore'] is not None))
    return pd.DataFrame(rows)

def _clean_cols(df):
    df.columns = [str(c).replace('ï»¿', '').replace('﻿', '').strip() for c in df.columns]
    return df

def load_domestic():
    frames = []
    for lg in MAIN_LEAGUES:
        for s in FD_SEASONS:
            p = os.path.join(DATA_DIR, f'{lg}_{s}.csv')
            if not os.path.exists(p) or os.path.getsize(p) < 200: continue
            df = _clean_cols(pd.read_csv(p, encoding='latin1', on_bad_lines='skip'))
            if not {'Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG'} <= set(df.columns): continue
            df = df.dropna(subset=['HomeTeam', 'AwayTeam', 'FTHG', 'FTAG'])
            frames.append(pd.DataFrame({
                'date': pd.to_datetime(df['Date'], dayfirst=True, format='mixed', errors='coerce'),
                'comp': lg, 'season_id': f'{lg}-{s}',
                'home': df['HomeTeam'].str.strip(), 'away': df['AwayTeam'].str.strip(),
                'hg': df['FTHG'], 'ag': df['FTAG'],
                'hst': df['HST'] if 'HST' in df else np.nan, 'ast': df['AST'] if 'AST' in df else np.nan}))
    for lg in EXTRA_LEAGUES:
        p = os.path.join(DATA_DIR, f'NEW_{lg}.csv')
        if not os.path.exists(p): continue
        df = _clean_cols(pd.read_csv(p, encoding='latin1', on_bad_lines='skip')).dropna(subset=['Home', 'Away', 'HG', 'AG'])
        d = pd.to_datetime(df['Date'], dayfirst=True, format='mixed', errors='coerce')
        frames.append(pd.DataFrame({
            'date': d, 'comp': lg, 'season_id': lg + '-' + df['Season'].astype(str),
            'home': df['Home'].str.strip(), 'away': df['Away'].str.strip(),
            'hg': df['HG'], 'ag': df['AG'], 'hst': np.nan, 'ast': np.nan})[d >= pd.Timestamp('2016-07-01')])
    dom = pd.concat(frames, ignore_index=True).dropna(subset=['date'])
    dom['round'] = 0; dom['neutral'] = False; dom['is_ko'] = False; dom['is_final'] = False; dom['winner'] = None; dom['played'] = True
    return dom

ucl = load_ucl(); dom = load_domestic()
matches = pd.concat([dom, ucl], ignore_index=True).sort_values('date', kind='stable').reset_index(drop=True)
matches[['hst', 'ast']] = matches[['hst', 'ast']].apply(pd.to_numeric, errors='coerce')
print(f'domestic: {len(dom):,} matches | UCL: {ucl.played.sum():,} played, {(~ucl.played).sum()} upcoming')
print(dom.groupby('comp').date.agg(['count', 'min', 'max']))

dom_teams = set(dom.home) | set(dom.away)
cur = ucl[ucl.season_id == f'UCL-{CURRENT_SEASON}']
no_dom = sorted((set(cur.home) | set(cur.away)) - dom_teams)
print('\n2026-27 UCL clubs without domestic data (rated from UCL results only):', no_dom)

## 3. Club ratings (Elo) + pre-match features
One chronological pass over **all** matches. For each match we first record features using only information available before kick-off, then update the state.

* **Elo**: K=20 domestic, 30 UCL; home advantage 65; goal-margin multiplier; 20% regression to league mean each new season. New leagues start at a prior roughly reflecting UEFA strength; promoted clubs start 100 below their league mean. UCL games are what move whole leagues relative to each other.
* **Form**: last 10 domestic games: points/game, goal diff/game, shots-on-target share.
* **Season PPG**: current domestic season points per game.
* **League strength**: mean Elo of the club's current league.
* **UCL experience / pedigree**: matches, PPG, knockout matches, finals and titles in the previous 3 UCL seasons.

In [ ]:
LEAGUE_PRIOR = {'E0': 1650, 'SP1': 1630, 'D1': 1610, 'I1': 1610, 'F1': 1570, 'P1': 1530, 'N1': 1510,
                'B1': 1480, 'T1': 1470, 'G1': 1450, 'SC0': 1440, 'AUT': 1450, 'DNK': 1440, 'NOR': 1420, 'SWZ': 1430}
UCL_ONLY_PRIOR = 1450
HFA, K_DOM, K_UCL, REGRESS, FORM_N = 65, 20, 30, 0.8, 10

def season_year(d):  # football season start year
    return d.year if d.month >= 7 else d.year - 1

class Engine:
    def __init__(self):
        self.elo = {}; self.league_of = {}; self.team_sid = {}
        self.members = defaultdict(set); self.league_mean = {}; self.cur_sid = {}; self.n_seasons = defaultdict(int)
        self.form = defaultdict(lambda: deque(maxlen=FORM_N))
        self.spts = defaultdict(lambda: [0, 0])
        self.ucl = defaultdict(lambda: defaultdict(lambda: [0, 0, 0, 0, 0]))   # team -> year -> [games, pts, ko games, titles, finals]

    def _new_season(self, lg, sid):
        m = self.members[lg]
        if m:
            mu = float(np.mean([self.elo[t] for t in m]))
            for t in m: self.elo[t] = mu + REGRESS * (self.elo[t] - mu)
            self.league_mean[lg] = mu
        else:
            self.league_mean[lg] = LEAGUE_PRIOR.get(lg, 1450)
        self.members[lg] = set(); self.cur_sid[lg] = sid; self.n_seasons[lg] += 1

    def _ensure(self, t, lg=None):
        if t in self.elo: return
        if lg is None: self.elo[t] = UCL_ONLY_PRIOR
        else: self.elo[t] = self.league_mean[lg] - (100 if self.n_seasons[lg] > 1 else 0)

    def prepare(self, m):
        if m.comp == 'UCL':
            self._ensure(m.home); self._ensure(m.away); return
        if self.cur_sid.get(m.comp) != m.season_id: self._new_season(m.comp, m.season_id)
        for t in (m.home, m.away):
            self._ensure(t, m.comp); self.members[m.comp].add(t)
            self.league_of[t] = m.comp; self.team_sid[t] = m.season_id

    def league_strength(self, lg):
        if lg is None: return np.nan
        m = self.members.get(lg)
        if m and len(m) >= 6: return float(np.mean([self.elo[t] for t in m]))
        return self.league_mean.get(lg, np.nan)

    def team_feats(self, t, date):
        f = self.form.get(t)
        if f:
            a = np.array(f, dtype=float)
            ppg, gd = a[:, 0].mean(), a[:, 1].mean()
            sot = np.nanmean(a[:, 2]) if np.isfinite(a[:, 2]).any() else np.nan
        else:
            ppg = gd = sot = np.nan
        sid = self.team_sid.get(t)
        sp = self.spts.get((t, sid)) if sid else None
        sppg = sp[0] / sp[1] if sp and sp[1] >= 3 else np.nan
        y = season_year(date); g = p = k = ti = fi = 0
        hist = self.ucl.get(t)
        if hist:
            for s in range(y - 3, y):
                if s in hist: g += hist[s][0]; p += hist[s][1]; k += hist[s][2]; ti += hist[s][3]; fi += hist[s][4]
        return dict(elo=self.elo.get(t, UCL_ONLY_PRIOR), ppg=ppg, gd=gd, sot=sot, sppg=sppg,
                    lstr=self.league_strength(self.league_of.get(t)),
                    ucl_g=g, ucl_ppg=p / g if g else np.nan, ucl_ko=k, ucl_titles=ti, ucl_finals=fi)

    def match_feats(self, h, a, date, neutral, is_ko, is_ucl):
        fh, fa = self.team_feats(h, date), self.team_feats(a, date)
        row = {}
        for k in fh:
            row[f'h_{k}'] = fh[k]; row[f'a_{k}'] = fa[k]; row[f'd_{k}'] = fh[k] - fa[k]
        row['is_home'] = 0 if neutral else 1; row['is_ko'] = int(is_ko)
        row['is_ucl'] = int(is_ucl); row['home_x_ucl'] = row['is_home'] * row['is_ucl']
        return row

    def update(self, m):
        h, a, hg, ag = m.home, m.away, int(m.hg), int(m.ag)
        is_ucl = m.comp == 'UCL'
        dr = self.elo[h] + (0 if m.neutral else HFA) - self.elo[a]
        exp_h = 1 / (1 + 10 ** (-dr / 400))
        s_h = 1.0 if hg > ag else 0.5 if hg == ag else 0.0
        gdiff = abs(hg - ag)
        mult = 1 if gdiff <= 1 else 1.5 if gdiff == 2 else (11 + gdiff) / 8
        delta = (K_UCL if is_ucl else K_DOM) * mult * (s_h - exp_h)
        self.elo[h] += delta; self.elo[a] -= delta
        ph = 3 if hg > ag else 1 if hg == ag else 0
        pa = 3 if ag > hg else 1 if hg == ag else 0
        if is_ucl:
            y = season_year(m.date)
            for t, pts in ((h, ph), (a, pa)):
                r = self.ucl[t][y]; r[0] += 1; r[1] += pts; r[2] += int(m.is_ko); r[4] += int(m.is_final)
            if m.is_final:
                w = m.winner if m.winner in (h, a) else (h if hg > ag else a if ag > hg else None)
                if w: self.ucl[w][y][3] += 1
        else:
            tot = (m.hst or 0) + (m.ast or 0)
            sh = m.hst / tot if np.isfinite(m.hst) and np.isfinite(m.ast) and tot > 0 else np.nan
            self.form[h].append((ph, hg - ag, sh)); self.form[a].append((pa, ag - hg, 1 - sh if np.isfinite(sh) else np.nan))
            for t, pts in ((h, ph), (a, pa)):
                r = self.spts[(t, m.season_id)]; r[0] += pts; r[1] += 1

eng = Engine()
played = matches[matches.played].reset_index(drop=True)
rows = []
for m in played.itertuples(index=False):
    eng.prepare(m)
    rows.append(eng.match_feats(m.home, m.away, m.date, m.neutral, m.is_ko, m.comp == 'UCL'))
    eng.update(m)

R = pd.DataFrame(rows)
F = pd.concat([played, R.drop(columns=[c for c in R.columns if c in played.columns])], axis=1)
F['is_ko'] = F.is_ko.astype(int)
F['hg'] = F.hg.astype(int); F['ag'] = F.ag.astype(int)
F['y'] = np.select([F.hg > F.ag, F.hg == F.ag], [0, 1], 2)          # 0 home win, 1 draw, 2 away win
F['year'] = F.date.map(season_year)
print(f'feature table: {F.shape}')

cur_teams = sorted(set(cur.home) | set(cur.away))
print('\nCurrent Elo of 2026-27 UCL clubs:')
print(pd.Series({t: round(eng.elo[t]) for t in cur_teams}).sort_values(ascending=False).to_string())

## 3b. ClubElo ratings (scraped from clubelo.com)
The ClubElo **API** (`api.clubelo.com`, used by `soccerdata` and by the C# `ClubElo.API` library) currently returns *502 Bad Gateway*, but the **website** still serves the same ratings:
* `clubelo.com/Ranking` → current Elo of ~1,600 clubs,
* `clubelo.com/<club>` → that club's rating history since late 2022.

Pages are cached for a day (on Drive if mounted) and fetched politely (0.5 s apart). For each past UCL match we use the club's ClubElo rating **from before the match date** (no leakage). If the site is unreachable the notebook carries on with its own Elo only.

In [ ]:
import time, re, html, unicodedata, difflib
CLUBELO_DIR = os.path.join(BASE_DIR, 'clubelo'); os.makedirs(CLUBELO_DIR, exist_ok=True)
HDRS = {'User-Agent': 'ucl-predictor/1.0 (personal research notebook)'}

def _get_cached(url, fname, max_age_days=1):
    path = os.path.join(CLUBELO_DIR, fname)
    if os.path.exists(path) and time.time() - os.path.getmtime(path) < max_age_days * 86400:
        with open(path, encoding='utf-8') as fh: return fh.read()
    r = requests.get(url, timeout=60, headers=HDRS); r.raise_for_status()
    with open(path, 'w', encoding='utf-8') as fh: fh.write(r.text)
    time.sleep(0.5)
    return r.text

ROW_RE  = re.compile(r'alt="(\w+)"[^\]]*?<a href="/([^"]+)">([^<]+)<span[^>]*></span></a></td>\',\s*\'(\d+)\'')
HIST_RE = re.compile(r'"Date":\s*"(\d{4}-\d{2}-\d{2})T[^"]*",\s*"Elo":\s*([\d.]+)')

# our name -> ClubElo name, where they differ
CE_OVERRIDES = {
    'Ath Madrid': 'Atlético', 'Ath Bilbao': 'Athletic Club', 'Bayern Munich': 'Bayern München', 'Sp Lisbon': 'Sporting',
    'Sp Braga': 'Braga', 'PSV Eindhoven': 'PSV', 'Ein Frankfurt': 'Frankfurt', "M'gladbach": 'Gladbach',
    'FC Copenhagen': 'FC Kobenhavn', 'Buyuksehyr': 'Başakşehir', 'S. Bratislava': 'Slovan', 'Club Brugge': 'Brugge',
    'M. Haifa': 'Maccabi Haifa', 'Kairat Almaty': 'Kairat', 'Bodo/Glimt': 'Bodø/Glimt', 'Inter': 'Internazionale',
    'Sabah': 'Sabah FK', 'Fenerbahce': 'Fenerbahçe', 'Sociedad': 'Real Sociedad', 'Salzburg': 'RB Salzburg',
    'Malmo': 'Malmö', 'Plzen': 'Viktoria Plzeň', 'Viktoria Plzen': 'Viktoria Plzeň', 'CSKA Moscow': 'CSKA Moskva',
    'Spartak Moscow': 'Spartak Moskva', 'Krasnodar': 'FC Krasnodar', 'Schalke 04': 'Schalke', 'St. Gilloise': 'St Gillis',
    'Ludogorets': 'Razgrad', 'Qarabag': 'Karabakh Agdam', 'Pafos': 'Paphos', 'Lokomotiv Moskva': 'Lok Moskva',
}
_DROP = {'fc', 'cf', 'afc', 'sc', 'sk', 'fk', 'ac', 'as', 'ssc', 'cd', 'sv', 'bk', 'club', 'kv', 'cp', 'bc', 'vfb', 'vfl'}
def _norm(x):
    x = unicodedata.normalize('NFKD', str(x)).encode('ascii', 'ignore').decode().lower()
    return ' '.join(t for t in re.split(r'[^a-z0-9]+', x) if t and t not in _DROP)
def make_matcher(ce_names):
    ce_names = set(ce_names); by_norm = {}
    for c in sorted(ce_names, key=len): by_norm.setdefault(_norm(c), c)
    def match(t):
        o = CE_OVERRIDES.get(t)
        if o in ce_names: return o
        if t in ce_names: return t
        n = _norm(t)
        if n in by_norm: return by_norm[n]
        best = difflib.get_close_matches(n, list(by_norm), n=1, cutoff=0.85)
        return by_norm[best[0]] if best else None
    return match

CLUBELO_OK, CELO_NOW, CELO_HIST, NAME_MAP = False, None, {}, {}
try:
    page = _get_cached('https://clubelo.com/Ranking', 'ranking.html')
    ce_rows = [(html.unescape(n).strip(), slug, cc, float(e)) for cc, slug, n, e in ROW_RE.findall(page)]
    if len(ce_rows) < 100: raise ValueError(f'parsed only {len(ce_rows)} clubs - page layout may have changed')
    CE = pd.DataFrame(ce_rows, columns=['name', 'slug', 'country', 'elo']).drop_duplicates('name').set_index('name')
    CELO_NOW = CE.elo; CLUBELO_OK = True
    print(f'ClubElo website: {len(CE):,} clubs in the current ranking')
except Exception as e:
    print(f"ClubElo website unavailable ({type(e).__name__}: {str(e)[:150]}) -> using this notebook's own Elo only")

def celo_at(team, date):
    s = CELO_HIST.get(NAME_MAP.get(team))
    if s is None or s.empty: return np.nan
    i = s.index.searchsorted(date, side='left') - 1          # last rating strictly before the match day
    return float(s.iloc[i]) if i >= 0 else np.nan

if CLUBELO_OK:
    match_name = make_matcher(CE.index)
    NAME_MAP = {t: match_name(t) for t in sorted(set(ucl.home) | set(ucl.away))}
    recent = ucl[ucl.date >= pd.Timestamp('2022-12-01')]
    need = sorted({NAME_MAP[t] for t in set(recent.home) | set(recent.away) if NAME_MAP.get(t)})
    print(f'fetching rating histories for {len(need)} clubs (cached for a day)...')
    for i, name in enumerate(need):
        slug = CE.slug[name]
        try:
            pg = _get_cached(f'https://clubelo.com/{slug}', 'club_' + re.sub(r'\W', '_', slug) + '.html')
            pts = HIST_RE.findall(pg)
            s = pd.Series([float(e) for _, e in pts], index=pd.to_datetime([d for d, _ in pts]))
            CELO_HIST[name] = s[~s.index.duplicated(keep='last')].sort_index()
        except Exception as e:
            print(f'  history failed for {name}: {e}')
        if (i + 1) % 25 == 0: print(f'  {i + 1}/{len(need)}')
    print('UCL clubs since Dec 2022 with no ClubElo match:',
          sorted(t for t in set(recent.home) | set(recent.away) if not NAME_MAP.get(t)) or 'none')

def _celo(series, team):
    n = NAME_MAP.get(team)
    return float(series.get(n, np.nan)) if (series is not None and n is not None) else np.nan

def attach_celo(df, homes, aways, series):
    df = df.copy()
    df['h_celo'] = [_celo(series, t) for t in homes]; df['a_celo'] = [_celo(series, t) for t in aways]
    df['d_celo'] = df.h_celo - df.a_celo
    return df

is_u = (F.comp == 'UCL').values
F['h_celo'] = np.nan; F['a_celo'] = np.nan
if CLUBELO_OK:
    F.loc[is_u, 'h_celo'] = [celo_at(t, d) for t, d in zip(F.home[is_u], F.date[is_u])]
    F.loc[is_u, 'a_celo'] = [celo_at(t, d) for t, d in zip(F.away[is_u], F.date[is_u])]
F['d_celo'] = F.h_celo - F.a_celo
if CLUBELO_OK:
    rec = F[is_u & (F.date >= pd.Timestamp('2023-01-01'))]
    print(f"UCL matches since 2023 with ClubElo for both clubs: {rec.d_celo.notna().mean():.1%} of {len(rec)}")
    cmp_ = pd.DataFrame({'our_elo': [round(eng.elo[t]) for t in cur_teams],
                         'clubelo': [_celo(CELO_NOW, t) for t in cur_teams],
                         'clubelo_name': [NAME_MAP.get(t) for t in cur_teams]}, index=cur_teams)
    print('\n2026-27 clubs: our Elo vs ClubElo (today)')
    print(cmp_.sort_values('clubelo', ascending=False).to_string())

## 4. Train & evaluate match-outcome models
* **Validation**: train ≤ 2022-23, score on 2023-24 → used to *choose* the model.
* **Test**: train ≤ 2023-24, score on 2024-25, 2025-26 and the played part of 2026-27 → honest out-of-sample numbers.
* Metrics: log-loss (lower is better), ranked probability score (RPS, lower is better), accuracy.

In [ ]:
DIFF = ['d_elo', 'd_ppg', 'd_gd', 'd_sot', 'd_sppg', 'd_lstr', 'd_ucl_g', 'd_ucl_ppg', 'd_ucl_ko', 'd_ucl_titles', 'd_ucl_finals']
LR_FEATS = DIFF + ['is_home', 'is_ko', 'is_ucl', 'home_x_ucl']
LEVEL = [f'{s}_{k}' for s in 'ha' for k in ['elo', 'ppg', 'gd', 'sot', 'sppg', 'lstr', 'ucl_g', 'ucl_ppg', 'ucl_ko', 'ucl_titles', 'ucl_finals']]
GBM_FEATS = LR_FEATS + LEVEL

lr  = lambda C: (lambda: make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000)))
gbm = lambda: HistGradientBoostingClassifier(max_depth=3, learning_rate=0.04, max_iter=250, min_samples_leaf=40,
                                             l2_regularization=1.0, random_state=SEED)
# name -> (factory, features, training rows, handles NaN natively)
MODELS = {
    'Elo-only LR (UCL)':          (lr(1.0), ['d_elo', 'is_home', 'is_ko'], 'ucl', False),
    'Elo + pedigree LR (UCL)':    (lr(1.0), ['d_elo', 'd_ucl_titles', 'd_ucl_finals', 'd_ucl_ko', 'is_home', 'is_ko'], 'ucl', False),
    'Full LR (UCL)':              (lr(0.3), LR_FEATS, 'ucl', False),
    'Full LR (UCL + domestic)':   (lr(1.0), LR_FEATS, 'all', False),
    'GBM (UCL)':                  (gbm, GBM_FEATS, 'ucl', True),
    'GBM (UCL + domestic)':       (gbm, GBM_FEATS, 'all', True),
}
def Xof(df, feats, nan_ok):
    X = df[feats]
    return X if nan_ok else X.fillna(0)

def train_rows(max_year, kind):
    tr = F[(F.year <= max_year) & (F.date >= BURN_IN)]
    if kind == 'ucl': return tr[tr.comp == 'UCL']
    if kind == 'ucl_celo': return tr[(tr.comp == 'UCL') & tr.d_celo.notna()]
    return tr

def rps(P, y):
    O = np.eye(3)[y]
    return np.mean(0.5 * ((P[:, 0] - O[:, 0]) ** 2 + (P[:, :2].sum(1) - O[:, :2].sum(1)) ** 2))

def score(P, y):
    return dict(log_loss=log_loss(y, P, labels=[0, 1, 2]), rps=rps(P, y), accuracy=accuracy_score(y, P.argmax(1)))

def evaluate(max_train_year, eval_years):
    te = F[(F.comp == 'UCL') & F.year.isin(eval_years)]
    base = np.bincount(train_rows(max_train_year, 'ucl').y, minlength=3) / len(train_rows(max_train_year, 'ucl'))
    res = {'Base rates (no model)': score(np.tile(base, (len(te), 1)), te.y.values)}
    for name, (make, feats, kind, nan_ok) in MODELS.items():
        tr = train_rows(max_train_year, kind)
        mdl = make().fit(Xof(tr, feats, nan_ok), tr.y)
        res[name] = score(mdl.predict_proba(Xof(te, feats, nan_ok)), te.y.values)
    return pd.DataFrame(res).T.round(4), len(te)

val, n_val = evaluate(2022, [2023])
test_years = list(range(2024, CURRENT_SEASON + 1))
test, n_test = evaluate(2023, test_years)
print(f'VALIDATION (2023-24, {n_val} matches)'); print(val.sort_values('log_loss'))
print(f'\nTEST (2024-25 → now, {n_test} matches)'); print(test.sort_values('log_loss'))

BEST = val.drop('Base rates (no model)').log_loss.idxmin()
print(f'\nSelected by validation: {BEST}  → test log-loss {test.loc[BEST, "log_loss"]:.4f}, '
      f'accuracy {test.loc[BEST, "accuracy"]:.1%}')
pd.concat({'validation': val, 'test': test}, axis=1).to_csv(f'{OUT_DIR}/model_comparison.csv')

# --- ClubElo vs our Elo, on the period where ClubElo history exists (late 2022 onwards) ---
# Every candidate is trained and scored on exactly the same matches: train 2022-23/2023-24, choose on 2024-25,
# report on 2025-26 + 2026-27 so far.
if CLUBELO_OK:
    ERA = {
        'Our Elo (LR)':              (lr(1.0), ['d_elo', 'is_home', 'is_ko'], 'ucl_celo', False),
        'ClubElo (LR)':              (lr(1.0), ['d_celo', 'is_home', 'is_ko'], 'ucl_celo', False),
        'Our Elo + ClubElo (LR)':    (lr(1.0), ['d_elo', 'd_celo', 'is_home', 'is_ko'], 'ucl_celo', False),
        'ClubElo + pedigree (LR)':   (lr(1.0), ['d_celo', 'd_ucl_titles', 'd_ucl_finals', 'd_ucl_ko', 'is_home', 'is_ko'], 'ucl_celo', False),
    }
    def era_eval(max_train_year, eval_years):
        te = F[(F.comp == 'UCL') & F.year.isin(eval_years) & F.d_celo.notna()]
        res = {}
        for name, (mk, fts, knd, nok) in ERA.items():
            tr = train_rows(max_train_year, knd)
            res[name] = score(mk().fit(Xof(tr, fts, nok), tr.y).predict_proba(Xof(te, fts, nok)), te.y.values)
        return pd.DataFrame(res).T.round(4), len(te), len(train_rows(max_train_year, 'ucl_celo'))
    era_val, nv, ntr = era_eval(2023, [2024])
    era_test, nt, _ = era_eval(2024, [2025, 2026])
    print(f'\nCLUBELO ERA - validation 2024-25 ({nv} matches, trained on {ntr})'); print(era_val.sort_values('log_loss'))
    print(f'\nCLUBELO ERA - test 2025-26 → now ({nt} matches)'); print(era_test.sort_values('log_loss'))
    ERA_BEST = era_val.log_loss.idxmin()
    if ERA_BEST != 'Our Elo (LR)':
        MODELS[ERA_BEST] = ERA[ERA_BEST]; BEST = ERA_BEST
        print(f'\n→ {ERA_BEST} beats our own Elo on validation: it becomes the final model.')
    else:
        print(f'\n→ ClubElo does not beat our own Elo on validation: keeping {BEST}.')
    pd.concat({'era_validation': era_val, 'era_test': era_test}, axis=1).to_csv(f'{OUT_DIR}/clubelo_comparison.csv')

## 5. Final models (refit on everything played) + Poisson goals model
The goals model (used for scorelines and the tournament simulation) uses **exactly the same features and training rows as the chosen match model**, so single-match predictions and title odds are consistent.

In [ ]:
make, feats, kind, nan_ok = MODELS[BEST]
tr_all = train_rows(CURRENT_SEASON, kind)
clf = make().fit(Xof(tr_all, feats, nan_ok), tr_all.y)

if hasattr(clf, 'steps') and hasattr(clf[-1], 'coef_'):
    coef = pd.DataFrame(clf[-1].coef_.T, index=feats, columns=['home win', 'draw', 'away win'])
    print('Standardised coefficients (home win − away win shows what pushes the result toward the home side):')
    print(coef.assign(**{'home−away': coef['home win'] - coef['away win']}).sort_values('home−away', key=abs, ascending=False).round(3))

# Poisson goals model: expected home / away goals from the SAME features as the chosen match model
GOAL_FEATS, GOAL_KIND = list(feats), kind
print(f'\nFinal model: {BEST} | features: {GOAL_FEATS} | training rows: {len(tr_all):,}')
def fit_poisson(df):
    sc = StandardScaler().fit(df[GOAL_FEATS].fillna(0))
    X = sc.transform(df[GOAL_FEATS].fillna(0))
    return sc, PoissonRegressor(alpha=1e-4, max_iter=2000).fit(X, df.hg), PoissonRegressor(alpha=1e-4, max_iter=2000).fit(X, df.ag)

def lambdas(pm, df):
    sc, mh, ma = pm; X = sc.transform(df[GOAL_FEATS].fillna(0))
    return mh.predict(X), ma.predict(X)

G = np.arange(11)
def hda_from_lambdas(lh, la):
    M = poisson.pmf(G[None, :, None], np.asarray(lh)[:, None, None]) * poisson.pmf(G[None, None, :], np.asarray(la)[:, None, None])
    return np.stack([np.tril(M, -1).sum((1, 2)), np.trace(M, axis1=1, axis2=2), np.triu(M, 1).sum((1, 2))], 1)

# out-of-sample check of the goals model on the same test window
pm_tmp = fit_poisson(train_rows(2023, GOAL_KIND))
te = F[(F.comp == 'UCL') & F.year.isin(test_years)]
print('\nPoisson goals model, TEST:', {k: round(v, 4) for k, v in score(hda_from_lambdas(*lambdas(pm_tmp, te)), te.y.values).items()})

pm = fit_poisson(train_rows(CURRENT_SEASON, GOAL_KIND))

## 6. Next UCL matches: who wins?

In [ ]:
TODAY = max(pd.Timestamp.today().normalize(), F.date.max())
fixtures = ucl[(~ucl.played) & (ucl.season_id == f'UCL-{CURRENT_SEASON}')].sort_values(['date', 'home']).reset_index(drop=True)
FX = pd.DataFrame([eng.match_feats(r.home, r.away, TODAY, r.neutral, r.is_ko, True) for r in fixtures.itertuples()])
FX = attach_celo(FX, fixtures.home, fixtures.away, CELO_NOW)
P = clf.predict_proba(Xof(FX, feats, nan_ok))
lh, la = lambdas(pm, FX)

def likely_score(a, b):
    M = np.outer(poisson.pmf(G, a), poisson.pmf(G, b)); i, j = np.unravel_index(M.argmax(), M.shape); return f'{i}-{j}'

preds = fixtures[['date', 'round', 'home', 'away']].assign(
    p_home=P[:, 0], p_draw=P[:, 1], p_away=P[:, 2], xg_home=lh, xg_away=la)
preds['prediction'] = np.select([preds.p_home >= preds[['p_draw', 'p_away']].max(1), preds.p_away >= preds.p_draw],
                                [preds.home + ' win', preds.away + ' win'], 'Draw')
preds['likely_score'] = [likely_score(a, b) for a, b in zip(lh, la)]
preds.to_csv(f'{OUT_DIR}/upcoming_predictions.csv', index=False)

nxt = preds[preds['round'] == preds['round'].min()]
print(f'NEXT MATCHDAY (MD{int(nxt["round"].iloc[0])})')
fmt = nxt.copy()
for c in ['p_home', 'p_draw', 'p_away']: fmt[c] = (fmt[c] * 100).round(0).astype(int).astype(str) + '%'
for c in ['xg_home', 'xg_away']: fmt[c] = fmt[c].round(2)
fmt['date'] = fmt.date.dt.strftime('%a %d %b')
display(fmt.drop(columns='round'))

## 7. Who will be champion? State-aware Monte Carlo of the rest of 2026-27
The simulation starts from **the real state of the competition** and only simulates what is still unknown:
* **League table** is built from league-phase matches only (rounds 1-8); remaining league games are simulated with the goals model; ranking = points → goal difference → goals scored (→ random).
* **Knockout ties already drawn** (they appear in the fixture feed after each UEFA draw) are used exactly as drawn. **Finished ties are locked** (aggregate, then extra time, then the real penalty winner). **Half-played ties** only simulate the 2nd leg, starting from the real 1st-leg score.
* **Ties not drawn yet** follow UEFA's rules (checked against the real 2025-26 bracket): play-offs 9/10 v 23/24, 11/12 v 21/22, 13/14 v 19/20, 15/16 v 17/18; R16 seeds 1/2 v winners from 15-18, 3/4 v 13-14/19-20, 5/6 v 11-12/21-22, 7/8 v 9-10/23-24; QF (1/2 path v 7/8 path) and (3/4 v 5/6); seeds of each pair in opposite halves.
* **Bracket halves**: which half each seed lands in is decided by UEFA's R16 draw and isn't in the fixture feed. Until you set `BRACKET_HALF_A` (the 4 seeded clubs in one half, after the draw) it's randomised in every run.
* Better-ranked club hosts the 2nd leg; extra time = 1/3 of the goal rate; penalties 50/50; final at a neutral venue. Ratings and form are frozen at today's values.

In [ ]:
BRACKET_HALF_A = None     # after UEFA's R16 draw, e.g. ['Arsenal', 'Sp Lisbon', 'Tottenham', 'Barcelona']
KO_STAGES = [('playoff', 9, 10), ('r16', 11, 12), ('qf', 13, 14), ('sf', 15, 16), ('final', 17, None)]
PO_GROUPS = [(8, 9, 22, 23), (10, 11, 20, 21), (12, 13, 18, 19), (14, 15, 16, 17)]   # 0-based league ranks
N = N_SIMS

def pair_matrices(teams, date):
    """Expected goals for every ordered pair: league game (L), knockout at home (K), knockout neutral (N)."""
    T = len(teams); pairs = [(i, j) for i in range(T) for j in range(T) if i != j]
    ii = np.array([p[0] for p in pairs]); jj = np.array([p[1] for p in pairs]); out = {}
    for key, neutral, ko in [('L', False, False), ('K', False, True), ('N', True, True)]:
        X = pd.DataFrame([eng.match_feats(teams[i], teams[j], date, neutral, ko, True) for i, j in pairs])
        X = attach_celo(X, [teams[i] for i in ii], [teams[j] for j in jj], CELO_NOW)
        a, b = lambdas(pm, X); A = np.zeros((T, T)); B = np.zeros((T, T))
        A[ii, jj] = a; B[ii, jj] = b; out[key] = (A, B)
    return out

def known_ties(rows, idx, r1, r2):
    """Knockout ties of one round whose clubs are known. a hosts leg 1, b hosts leg 2."""
    leg1 = rows[(rows['round'] == r1) & rows.home.isin(list(idx)) & rows.away.isin(list(idx))]
    ties = []
    for r in leg1.itertuples():
        t = dict(a=idx[r.home], b=idx[r.away], l1=(int(r.hg), int(r.ag)) if r.played else None, l2=None, w2=None)
        if r2 is None:
            t['w2'] = r.winner if r.played else None
        else:
            m2 = rows[(rows['round'] == r2) & (rows.home == r.away) & (rows.away == r.home)]
            if len(m2) and bool(m2.played.iloc[0]):
                t['l2'] = (int(m2.hg.iloc[0]), int(m2.ag.iloc[0]))       # (b's goals at home, a's goals away)
                t['w2'] = m2.winner.iloc[0]
        ties.append(t)
    return ties

def simulate(rows, teams, mats, N=N_SIMS, seed=SEED, half_a=None):
    rng = np.random.default_rng(seed)
    T = len(teams); idx = {t: i for i, t in enumerate(teams)}; ar = np.arange(N)
    (LHL, LAL), (LH, LA), (NH, NA) = mats['L'], mats['K'], mats['N']

    # ---- league phase (rounds 1-8 only) ----
    lg = rows[rows['round'] <= 8]
    base = np.zeros((3, T))
    for r in lg[lg.played].itertuples():
        h, a, hg, ag = idx[r.home], idx[r.away], int(r.hg), int(r.ag)
        base[0, h] += 3 if hg > ag else 1 if hg == ag else 0; base[0, a] += 3 if ag > hg else 1 if hg == ag else 0
        base[1, h] += hg - ag; base[1, a] += ag - hg; base[2, h] += hg; base[2, a] += ag
    rem = lg[~lg.played]
    hi = rem.home.map(idx).values.astype(int); ai = rem.away.map(idx).values.astype(int)
    pts = np.tile(base[0], (N, 1)); gd = np.tile(base[1], (N, 1)); gf = np.tile(base[2], (N, 1))
    if len(hi):
        gh = rng.poisson(LHL[hi, ai], size=(N, len(hi))); ga = rng.poisson(LAL[hi, ai], size=(N, len(hi)))
        hp = np.where(gh > ga, 3, np.where(gh == ga, 1, 0)); ap = np.where(ga > gh, 3, np.where(gh == ga, 1, 0))
        for k in range(len(hi)):
            pts[:, hi[k]] += hp[:, k]; pts[:, ai[k]] += ap[:, k]
            gd[:, hi[k]] += gh[:, k] - ga[:, k]; gd[:, ai[k]] += ga[:, k] - gh[:, k]
            gf[:, hi[k]] += gh[:, k]; gf[:, ai[k]] += ga[:, k]
    order = np.lexsort((rng.random((N, T)), -gf, -gd, -pts), axis=-1)     # best first
    rank_of = np.empty_like(order); rank_of[ar[:, None], order] = np.arange(T)[None, :]
    seat = lambda k: order[:, k]
    flip = lambda: rng.random(N) < 0.5

    # ---- tie helpers ----
    def two_legs(A, B, l1=None, l2=None, w2=None):           # A hosts leg 1, B hosts leg 2
        A = np.broadcast_to(A, (N,)); B = np.broadcast_to(B, (N,))
        if l1 is not None: a1 = np.full(N, l1[0]); b1 = np.full(N, l1[1])
        else: a1 = rng.poisson(LH[A, B]); b1 = rng.poisson(LA[A, B])
        if l2 is not None:                                     # tie finished: lock it
            ga_, gb_ = a1 + l2[1], b1 + l2[0]
            pen_a = np.full(N, w2 == teams[A[0]]) if w2 in idx else flip()
            return np.where(ga_ > gb_, A, np.where(gb_ > ga_, B, np.where(pen_a, A, B)))
        b2 = rng.poisson(LH[B, A]); a2 = rng.poisson(LA[B, A])
        ga_, gb_ = a1 + a2, b1 + b2; tied = ga_ == gb_
        ga_ = ga_ + tied * rng.poisson(LA[B, A] / 3); gb_ = gb_ + tied * rng.poisson(LH[B, A] / 3)
        return np.where((ga_ > gb_) | ((ga_ == gb_) & flip()), A, B)

    def seeded_tie(X, Y):                                      # better-ranked club hosts the 2nd leg
        xb = rank_of[ar, X] < rank_of[ar, Y]
        return two_legs(np.where(xb, Y, X), np.where(xb, X, Y))

    play_known = lambda t: two_legs(np.full(N, t['a']), np.full(N, t['b']), t['l1'], t['l2'], t['w2'])

    def final_match(X, Y, known=None):
        X = np.broadcast_to(X, (N,)); Y = np.broadcast_to(Y, (N,))
        if known is not None and known['l1'] is not None:
            gx, gy = known['l1']
            if gx != gy: return np.full(N, known['a'] if gx > gy else known['b'])
            return np.full(N, idx[known['w2']]) if known['w2'] in idx else np.where(flip(), X, Y)
        gx = rng.poisson(NH[X, Y]); gy = rng.poisson(NA[X, Y]); t = gx == gy
        gx = gx + t * rng.poisson(NH[X, Y] / 3); gy = gy + t * rng.poisson(NA[X, Y] / 3)
        return np.where((gx > gy) | ((gx == gy) & flip()), X, Y)

    kn = {st: known_ties(rows, idx, r1, r2) for st, r1, r2 in KO_STAGES}
    seed_rank = lambda t: int(min(rank_of[0, t['a']], rank_of[0, t['b']]))   # league is final once ties are drawn

    # ---- knockout play-offs ----
    po_w = [[None, None] for _ in range(4)]
    if len(kn['playoff']) == 8:
        for t in kn['playoff']:
            s = seed_rank(t); po_w[(s - 8) // 2][s % 2] = play_known(t)
    else:
        for g, (s1, s2, u1, u2) in enumerate(PO_GROUPS):
            f = flip()
            po_w[g] = [seeded_tie(seat(s1), np.where(f, seat(u1), seat(u2))), seeded_tie(seat(s2), np.where(f, seat(u2), seat(u1)))]

    # ---- round of 16 ----
    r16_w = [[None, None] for _ in range(4)]
    if len(kn['r16']) == 8:
        for t in kn['r16']:
            s = seed_rank(t); r16_w[s // 2][s % 2] = play_known(t)
        r16_teams = np.tile([t['a'] for t in kn['r16']] + [t['b'] for t in kn['r16']], (N, 1))
    else:
        for k in range(4):
            w1, w2 = po_w[3 - k]; f = flip()
            r16_w[k] = [seeded_tie(seat(2 * k), np.where(f, w1, w2)), seeded_tie(seat(2 * k + 1), np.where(f, w2, w1))]
        r16_teams = np.concatenate([order[:, :8]] + [np.stack(p, 1) for p in po_w], 1)

    # ---- bracket halves: seed 2k's path goes to half A when inA[k] ----
    if half_a:
        inA = [np.full(N, teams[order[0, 2 * k]] in half_a) for k in range(4)]
    else:
        inA = [flip() for _ in range(4)]
    halves = [[np.where(inA[k], r16_w[k][0], r16_w[k][1]) for k in range(4)],
              [np.where(inA[k], r16_w[k][1], r16_w[k][0]) for k in range(4)]]

    # ---- quarter-finals and semi-finals ----
    if len(kn['qf']) == 4:
        qf_res = [(t, play_known(t)) for t in kn['qf']]
        qf_w = [w for _, w in qf_res]
        if len(kn['sf']) == 2:
            sf_w = [play_known(t) for t in kn['sf']]
        else:
            path = {}                                            # club -> seed rank of its R16 tie
            for t in kn['r16']: path[t['a']] = path[t['b']] = seed_rank(t)
            outer = [(t, w) for t, w in qf_res if path.get(t['a'], 9) // 2 in (0, 3)]
            inner = [(t, w) for t, w in qf_res if path.get(t['a'], 9) // 2 in (1, 2)]
            def in_half_a(t, pairs):                             # is this QF tie in half A?
                s = [path[c] for c in (t['a'], t['b']) if path[c] // 2 in pairs][0]
                return teams[order[0, s]] in half_a              # a seed's path stays in that seed's half
            if len(outer) != 2 or len(inner) != 2:
                print('warning: could not place quarter-finals in the bracket; pairing them in feed order')
                sf_w = [seeded_tie(qf_w[0], qf_w[1]), seeded_tie(qf_w[2], qf_w[3])]
            elif half_a:
                o = sorted(outer, key=lambda x: not in_half_a(x[0], (0,))); i_ = sorted(inner, key=lambda x: not in_half_a(x[0], (1,)))
                sf_w = [seeded_tie(o[0][1], i_[0][1]), seeded_tie(o[1][1], i_[1][1])]
            else:
                f = flip()
                sf_w = [seeded_tie(outer[0][1], np.where(f, inner[0][1], inner[1][1])),
                        seeded_tie(outer[1][1], np.where(f, inner[1][1], inner[0][1]))]
    else:
        qf_w, sf_w = [], []
        for H in halves:
            q1, q2 = seeded_tie(H[0], H[3]), seeded_tie(H[1], H[2]); qf_w += [q1, q2]
            sf_w.append(seeded_tie(q1, q2) if len(kn['sf']) != 2 else None)
        if len(kn['sf']) == 2: sf_w = [play_known(t) for t in kn['sf']]

    # ---- final ----
    if len(kn['final']) == 1:
        t = kn['final'][0]; champ = final_match(np.full(N, t['a']), np.full(N, t['b']), t)
    else:
        champ = final_match(sf_w[0], sf_w[1])

    def freq(arr):
        c = np.zeros(T); np.add.at(c, np.asarray(arr).ravel(), 1); return c / N
    out = pd.DataFrame({
        'pts_now': base[0].astype(int), 'exp_pts': pts.mean(0).round(1),
        'top8': freq(order[:, :8]), 'playoff_9_24': freq(order[:, 8:24]),
        'R16': freq(r16_teams), 'QF': freq(np.stack([x for p in r16_w for x in p], 1)),
        'SF': freq(np.stack(qf_w, 1)), 'final': freq(np.stack(sf_w, 1)), 'champion': freq(champ)}, index=teams)
    out.index.name = 'club'
    assert abs(out.champion.sum() - 1) < 1e-9 and abs(out.final.sum() - 2) < 1e-9
    return out.sort_values(['champion', 'final', 'SF', 'exp_pts'], ascending=False)

teams = cur_teams
MATS = pair_matrices(teams, TODAY)
summary = simulate(cur, teams, MATS, half_a=BRACKET_HALF_A)
summary['elo'] = [round(eng.elo[t]) for t in summary.index]
summary['clubelo'] = [_celo(CELO_NOW, t) for t in summary.index]
summary.to_csv(f'{OUT_DIR}/champion_probabilities.csv')
known_now = {st: len(known_ties(cur, {t: i for i, t in enumerate(teams)}, r1, r2)) for st, r1, r2 in KO_STAGES}
print('knockout ties already known from the feed:', known_now)

show = summary.copy()
for c in ['top8', 'playoff_9_24', 'R16', 'QF', 'SF', 'final', 'champion']: show[c] = (show[c] * 100).round(1)
print(f'{N:,} simulations — probabilities in %')
display(show)

In [ ]:
top = summary.head(16)[::-1]
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top.index, top.champion * 100, color='#2b5fd9')
for y, v in enumerate(top.champion * 100): ax.text(v + 0.3, y, f'{v:.1f}%', va='center', fontsize=9)
ax.set_xlabel('Probability of winning the 2026-27 Champions League (%)')
ax.set_title(f'UCL 2026-27 title odds ({N:,} simulations, as of {TODAY:%d %b %Y})')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/champion_odds.png', dpi=150); plt.show()
print(f'Most likely champion: {summary.index[0]} ({summary.champion.iloc[0]:.1%})')

### 7b. Replay test: does the knockout logic follow reality? (2025-26)
Re-runs the simulator on last season "as of" several dates, hiding everything after that date. A knockout round counts as drawn 14 days before its first leg, once the previous round is finished. This checks the **mechanics** (table, real draws, locked ties, half-played ties, bracket halves), not prediction accuracy: ratings are today's.

In [ ]:
def replay_rows(season_rows, cutoff, draw_lead_days=14):
    r = season_rows.copy()
    r['played'] = r.played & (r.date <= cutoff)
    r.loc[~r.played, ['hg', 'ag', 'winner']] = None
    keep = r['round'] <= 8
    for st, r1, r2 in KO_STAGES:
        first_leg = season_rows[season_rows['round'] == r1].date.min()
        drawn = cutoff >= first_leg - pd.Timedelta(days=draw_lead_days) and bool(r[r['round'] < r1].played.all())
        if drawn: keep |= r['round'].isin([r1] if r2 is None else [r1, r2])
    return r[keep]

prev = ucl[ucl.season_id == f'UCL-{CURRENT_SEASON - 1}']
teams_prev = sorted(set(prev.home) | set(prev.away))
mats_prev = pair_matrices(teams_prev, TODAY)
asof = lambda d, half=None: simulate(replay_rows(prev, pd.Timestamp(d)), teams_prev, mats_prev, N=5000, half_a=half)

checks = []
def check(name, ok): checks.append((name, 'PASS' if ok else 'FAIL'))
TOP8 = ['Arsenal', 'Bayern Munich', 'Liverpool', 'Tottenham', 'Barcelona', 'Chelsea', 'Sp Lisbon', 'Man City']
R16 = TOP8 + ['Galatasaray', 'Atalanta', 'Newcastle', 'Ath Madrid', 'Leverkusen', 'Paris SG', 'Real Madrid', 'Bodo/Glimt']
SF4 = ['Paris SG', 'Bayern Munich', 'Ath Madrid', 'Arsenal']
HALF_A_2025 = ['Arsenal', 'Sp Lisbon', 'Tottenham', 'Barcelona']        # real 2025-26 bracket half

s = asof('2026-01-29')                                                   # league phase over, nothing drawn
check('after league phase: real top 8 qualify with 100%', (s.loc[TOP8, 'top8'] == 1).all() and s.top8.sum() == 8)
s = asof('2026-03-01')                                                   # play-offs done, R16 drawn
check('after play-offs: exactly the real 16 clubs in R16', (s.loc[R16, 'R16'] == 1).all() and abs(s.R16.sum() - 16) < 1e-9)
s = asof('2026-03-12')                                                   # R16 first legs played
check('half-played tie: Bayern (6-1 up) reach QF > 95%', s.loc['Bayern Munich', 'QF'] > 0.95)
check('half-played tie: Atalanta (1-6 down) reach QF < 5%', s.loc['Atalanta', 'QF'] < 0.05)
s0 = asof('2026-03-19'); s1 = asof('2026-03-19', HALF_A_2025)            # R16 done, QF not drawn yet
check('after R16: the 8 real R16 losers are locked out',
      (s1.loc[['Galatasaray', 'Man City', 'Chelsea', 'Tottenham', 'Leverkusen', 'Newcastle', 'Bodo/Glimt', 'Atalanta'], 'QF'] == 0).all())
check('with the real bracket: Arsenal and Sporting meet in a QF (exactly one reaches SF)',
      abs(s1.loc['Arsenal', 'SF'] + s1.loc['Sp Lisbon', 'SF'] - 1) < 1e-9)
check('without the bracket: that QF is not forced', abs(s0.loc['Arsenal', 'SF'] + s0.loc['Sp Lisbon', 'SF'] - 1) > 1e-6)
s = asof('2026-04-16')                                                   # QF done, SF drawn
check('after QF: the real four semi-finalists at 100%', (s.loc[SF4, 'SF'] == 1).all() and abs(s.SF.sum() - 4) < 1e-9)
check('after QF: PSG v Bayern is a semi-final (exactly one reaches the final)',
      abs(s.loc['Paris SG', 'final'] + s.loc['Bayern Munich', 'final'] - 1) < 1e-9)
s = asof('2026-04-30')                                                   # SF first legs played
check('SF first leg PSG 5-4 Bayern: PSG favoured to reach final', s.loc['Paris SG', 'final'] > s.loc['Bayern Munich', 'final'])
s = asof('2026-05-31')                                                   # everything played
check('after the final: PSG champion with 100%', s.loc['Paris SG', 'champion'] == 1)
print(pd.DataFrame(checks, columns=['check', 'result']).to_string(index=False))
print('\nALL REPLAY CHECKS PASSED' if all(r == 'PASS' for _, r in checks) else '\nSOME REPLAY CHECKS FAILED')

### 7c. Export for the web dashboard
Writes `dashboard_data.json` (model coefficients, club ratings, fixtures, results, knockout state, simulation summary) — everything the dashboard needs to recompute predictions and re-run the simulation in the browser. It is also printed in compressed chunks (`DASH|…`) so it can be copied out of the notebook.

In [ ]:
import zlib, base64, math
sc_, mh_, ma_ = pm
keys = sorted({f[2:] for f in GOAL_FEATS if f.startswith('d_')})
def clean(v):
    return None if v is None or (isinstance(v, float) and (math.isnan(v) or math.isinf(v))) else (round(float(v), 4) if isinstance(v, (float, np.floating)) else v)
def team_vals(t):
    f = eng.team_feats(t, TODAY); f['celo'] = _celo(CELO_NOW, t)
    return {k: clean(f.get(k)) for k in keys}
tidx = {t: i for i, t in enumerate(teams)}
lg_rows = cur[cur['round'] <= 8]
dash = {
    'generated': str(TODAY.date()), 'season': f'{CURRENT_SEASON}-{(CURRENT_SEASON + 1) % 100:02d}',
    'model': BEST, 'features': GOAL_FEATS,
    'test_logloss': clean(test.loc[BEST, 'log_loss']) if BEST in test.index else None,
    'test_accuracy': clean(test.loc[BEST, 'accuracy']) if BEST in test.index else None,
    'goals': {'mean': [clean(x) for x in sc_.mean_], 'scale': [clean(x) for x in sc_.scale_],
              'home_coef': [clean(x) for x in mh_.coef_], 'home_icpt': clean(mh_.intercept_),
              'away_coef': [clean(x) for x in ma_.coef_], 'away_icpt': clean(ma_.intercept_)},
    'teams': [{'name': t, 'elo': clean(eng.elo[t]), 'clubelo': clean(_celo(CELO_NOW, t)), 'vals': team_vals(t)} for t in teams],
    'played': [[int(r['round']), tidx[r.home], tidx[r.away], int(r.hg), int(r.ag)] for _, r in lg_rows[lg_rows.played].iterrows()],
    'left': [[int(r['round']), str(r.date.date()), tidx[r.home], tidx[r.away]] for _, r in lg_rows[~lg_rows.played].iterrows()],
    'ko': {st: [[t['a'], t['b'], t['l1'], t['l2'], tidx.get(t['w2'])] for t in known_ties(cur, tidx, r1, r2)] for st, r1, r2 in KO_STAGES},
    'bracket_half_a': [tidx[t] for t in BRACKET_HALF_A] if BRACKET_HALF_A else None,
    'summary': {t: [clean(x) for x in summary.loc[t, ['top8', 'playoff_9_24', 'R16', 'QF', 'SF', 'final', 'champion']]] for t in teams},
}
js = json.dumps(dash, separators=(',', ':'), ensure_ascii=False)
with open(f'{OUT_DIR}/dashboard_data.json', 'w', encoding='utf-8') as fh: fh.write(js)
blob = base64.urlsafe_b64encode(zlib.compress(js.encode('utf-8'), 9)).decode().rstrip('=')
CH = 1200
print(f'dashboard_data.json: {len(js):,} bytes -> {len(blob):,} compressed chars, {math.ceil(len(blob) / CH)} chunks')
for i in range(0, len(blob), CH): print(f'DASH|{i // CH}|{blob[i:i + CH]}')

## 8. Download results (optional)

In [ ]:
import shutil
zip_path = shutil.make_archive(os.path.join(BASE_DIR, 'ucl_outputs'), 'zip', OUT_DIR)
print('outputs zipped to', zip_path)
AUTO_DOWNLOAD = False          # set True to also download the zip through the browser
if IN_COLAB and AUTO_DOWNLOAD:
    try:
        from google.colab import files
        files.download(zip_path)
    except Exception as e:
        print('browser download not available here:', e)